# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [ ]:
from rtm_pymmcore.microscope.MMDemo import MMDemo

mic = MMDemo(
    "Z:\\mic01-imaging\\Cedric\\experimental_data\\2025-05-19_ST-Model",
    micromanager_path="D:\\Program Files\\Micro-Manager-2.0_n",
)
mic.mmc.setChannelGroup("Channel")


# for Jungfrau Microscope enable here:
# from rtm_pymmcore.microscope.Jungfrau import Jungfrau
# mic = Jungfrau()
# mic.mmc.setChannelGroup("TTL_ERK")

In [3]:
## Configuration options
N_FRAMES = 1
FIRST_FRAME_STIMULATION = 1

SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # time in seconds between frames
TIME_PER_FOV = 3  # time in seconds per fov

## Storage path for the experiment
base_path = "C:\\Users\\micromanager\\Desktop\\Cedric"
experiment_name = "exp_test2"
path = os.path.join(base_path, experiment_name)
path_with_old_data_for_simulation = os.path.join(
    ".", "test_exp_data", "01_demo_imgs_w_optocheck"
)

# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(
    Channel(
        name="DAPI",
        exposure=150,
        group="Channel",
        power=2,
        device_name="LED",
        property_name="State",
    )
)
channels.append(Channel(name="Cy5", exposure=150))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="FITC", exposure=150, group="Channel")


# Condition mapping to FOVs. This is used to create a dataframe with the conditions and the FOVs.
condition = [
    "FGFR_high"
]  # Example of adding a condition to the dataframe. Stimulation will be repeated for each condition.
# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.

n_fovs_per_condition = None  ## change this variable to the amount of fovs that you have per cell line. If only one cell line is set, this value will
# automatically set to total amount of fovs.

n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures = [
    60
]  # or e.g. [10, 20, 30] for different exposures. The exposure time is the time that the LED is on.
# Define the stimulation timesteps
stim_timesteps = [
    range(FIRST_FRAME_STIMULATION, N_FRAMES, 2)
]  # Using range to define timesteps from FIRST_FRAME_STIMULATION to N_FRAMES with step 2

# Combine the different paramters in stim_exposure and stim timestep to create stim_treatments which represents all possible combinations
stim_treatments = [
    StimTreatment(
        stim_channel_name="FITC",
        stim_channel_group="Channel",
        stim_timestep=stim_timestep,
        stim_exposure=stim_exposure,
        stim_power=3,
        stim_channel_device_name="LED",
        stim_channel_power_property_name="State",
    )
    for stim_exposure in stim_exposures
    for stim_timestep in stim_timesteps
]
for stim_treatment in stim_treatments:
    if isinstance(stim_treatment.stim_timestep, range):
        stim_treatment.stim_timestep = tuple(stim_treatment.stim_timestep)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.fe_for_optocheck import OptoCheckFE

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            diameter=50,
            custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_v1\\LifeActH2B_mixed_v1",
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]
stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels", distance=5, margin=3)
tracker = TrackerTrackpy()
optocheck_fe = OptoCheckFE("labels")

pprint(stim_treatments)


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck_fe,
)
mic.set_pipeline(pipeline=pipeline)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.4 
platform:       	win32 
python version: 	3.11.0 
torch version:  	2.7.1+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


[StimTreatment(stim_channel_name='FITC',
               stim_channel_group='Channel',
               stim_timestep=(),
               stim_exposure=60,
               stim_power=3,
               stim_channel_device_name='LED',
               stim_channel_power_property_name='State')]
Directory C:\Users\micromanager\Desktop\Cedric\exp_test2\raw already exists
Directory C:\Users\micromanager\Desktop\Cedric\exp_test2\tracks already exists
Directory C:\Users\micromanager\Desktop\Cedric\exp_test2\stim_mask already exists
Directory C:\Users\micromanager\Desktop\Cedric\exp_test2\stim already exists
Directory C:\Users\micromanager\Desktop\Cedric\exp_test2\particles already exists
Directory C:\Users\

### GUI - Napari Micromanager

#### Load GUI

In [29]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

#### Functions to break and re-connect link with GUI if manually broken

The following functions can be used to manually interrupt to connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [30]:
### Break connection
# mm_wdg._core_link.cleanup()

In [31]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [4]:
import json

file = "fovs.json"

# file = os.path.join(path, "fovs.json")
with open(file, "r") as f:
    data_mda_fovs = json.load(f)
data_mda_fovs = data_mda_fovs
load_from_file = True

### Use FOVs to generate dataframe for acquisition

In [5]:
n_fovs_simultaneously = TIME_BETWEEN_TIMESTEPS // TIME_PER_FOV
timesteps = range(N_FRAMES)


start_time = 0
if not load_from_file:
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    if data_mda_fovs is None:
        assert False, "No fovs selected. Please select fovs in the MDA widget"

if "channel_optocheck" not in locals():
    channel_optocheck = None
dfs = []
fovs = []
for fov_index, fov in enumerate(data_mda_fovs):
    fov_object = Fov(fov_index)
    fovs.append(fov_object)
    fov_group = fov_index // n_fovs_simultaneously
    start_time = fov_group * TIME_BETWEEN_TIMESTEPS * len(timesteps)
    if len(condition) == 1:
        condition_fov = condition[0]
    else:
        condition_fov = condition[fov_index // n_fovs_per_condition]
    for timestep in timesteps:
        row = {
            "fov_object": fov_object,
            "fov": fov_index,
            "fov_x": fov.get("x"),
            "fov_y": fov.get("y"),
            "fov_z": fov.get("z") if not mic.USE_ONLY_PFS else None,
            "fov_name": str(fov_index) if fov["name"] is None else fov["name"],
            "timestep": timestep,
            "time": start_time + timestep * TIME_BETWEEN_TIMESTEPS,
            "cell_line": condition_fov,
            "channels": tuple(dataclasses.asdict(channel) for channel in channels),
            "fname": f"{str(fov_index).zfill(3)}_{str(timestep).zfill(5)}",
        }
        if channel_optocheck is not None:
            row["optocheck"] = True if timestep == N_FRAMES - 1 else False

            if isinstance(channel_optocheck, list):
                row["optocheck_channels"] = tuple(
                    dataclasses.asdict(channel) for channel in channel_optocheck
                )
            else:
                row["optocheck_channels"] = tuple(
                    [dataclasses.asdict(channel_optocheck)]
                )

        dfs.append(row)

df_acquire = pd.DataFrame(dfs)

print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")

for stim_treatment in stim_treatments:
    if isinstance(stim_treatment.stim_timestep, range):
        stim_treatment.stim_timestep = tuple(stim_treatment.stim_timestep)

n_fovs = len(data_mda_fovs)
n_stim_treatments = len(stim_treatments)
if n_stim_treatments > 0:
    n_fovs_per_stim_condition = n_fovs // n_stim_treatments // len(np.unique(condition))
    stim_treatment_tot = []
    random.shuffle(stim_treatments)
    if n_fovs_per_well is not None:
        for stim_treat in stim_treatments:
            stim_treatment_tot.extend([stim_treat] * n_fovs_per_well)

    else:
        for fov_index in range(0, n_fovs_per_stim_condition):
            stim_treatment_tot.extend(stim_treatments)
        random.shuffle(stim_treatment_tot)

        if n_fovs % n_stim_treatments != 0:
            print(
                f"Warning: Not equal number of fovs per stim condition. {n_fovs % n_stim_treatments} fovs will have repeated treatment"
            )
            stim_treatment_tot.extend(stim_treatments[: n_fovs % n_stim_treatments])
    print(f"Doing {n_fovs_per_stim_condition} experiment per stim condition")

    if len(condition) == 1:
        n_fovs_per_condition = n_fovs
    else:
        stim_treatment_tot = stim_treatment_tot * len(np.unique(condition))

    df_acquire = pd.merge(
        df_acquire, pd.DataFrame(stim_treatment_tot), left_on="fov", right_index=True
    )

    # Add stim column that checks if current timestep is in the stim_timestep tuple
    df_acquire["stim"] = df_acquire.apply(
        lambda row: (
            row["timestep"] in row["stim_timestep"]
            if isinstance(row["stim_timestep"], tuple) and row["stim_exposure"] > 0
            else False
        ),
        axis=1,
    )

df_acquire = df_acquire.dropna(axis=1, how="all")
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
df_acquire = df_acquire.sort_values(by=["time", "fov"]).reset_index(drop=True)
df_acquire

Total Experiment Time: 0.08194444444444444h
Doing 60 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,cell_line,channels,fname,optocheck,optocheck_channels,stim_channel_name,stim_channel_group,stim_timestep,stim_exposure,stim_power,stim_channel_device_name,stim_channel_power_property_name,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,0,0,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",000_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,0,5,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",001_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,0.00,0.0,0.0,2,0,10,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",002_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,3,20.01,0.0,0.0,3,0,15,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",003_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,4,0.00,0.0,0.0,4,0,20,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",004_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
5,<rtm_pymmcore.data_structures.Fov object at 0x...,5,20.01,0.0,0.0,5,0,25,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",005_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
6,<rtm_pymmcore.data_structures.Fov object at 0x...,6,0.00,0.0,0.0,6,0,30,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",006_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
7,<rtm_pymmcore.data_structures.Fov object at 0x...,7,20.01,0.0,0.0,7,0,35,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",007_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
8,<rtm_pymmcore.data_structures.Fov object at 0x...,8,0.00,0.0,0.0,8,0,40,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",008_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False
9,<rtm_pymmcore.data_structures.Fov object at 0x...,9,20.01,0.0,0.0,9,0,45,FGFR_high,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",009_00000,True,"({'name': 'FITC', 'exposure': 150, 'group': 'C...",FITC,Channel,(),60,3,LED,State,False


### Run experiment

In [6]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)


try:
    mm_wdg._core_link.cleanup()
except:
    pass


mic.run_experiment(df_acquire)
print("Experiment finished")
mic.post_experiment()

time.sleep(120)
fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

Experiment finished
